In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -q git+https://github.com/facebookresearch/EGG.git
# (only needed for the referential game baseline; our code doesn't use wrappers)

In [ ]:
import egg

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import json
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── Hyperparams ──────────────────────────────────────────────────────────────
N_FEATURES    = 16
N_DISTRACTORS = 9       # receiver picks 1 from 10 candidates
N_TRAIN       = 10000
N_VAL         = 2000
HIDDEN        = 64
VOCAB_SIZE    = 32      # message dimensionality (same for both protocols)
N_EPOCHS      = 60
BATCH_SIZE    = 64
LR            = 1e-3
NOISE_LEVELS  = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# ── Data ─────────────────────────────────────────────────────────────────────
def make_dataset(n_samples, seed=42):
    rng = np.random.default_rng(seed)
    # each sample: (n_distractors+1) binary vectors; target is always index 0
    objects = rng.integers(0, 2, (n_samples, N_DISTRACTORS+1, N_FEATURES)).astype(np.float32)
    sender_input   = torch.tensor(objects[:, 0, :])
    labels         = torch.zeros(n_samples, dtype=torch.long)
    receiver_input = torch.tensor(objects)   # (N, 10, 16)
    return TensorDataset(sender_input, labels, receiver_input)

train_ds = make_dataset(N_TRAIN, seed=42)
val_ds   = make_dataset(N_VAL,   seed=99)

# ── Agents ───────────────────────────────────────────────────────────────────
class Sender(nn.Module):
    def __init__(self, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(N_FEATURES, HIDDEN), nn.ReLU(),
            nn.Linear(HIDDEN, output_dim),
        )
    def forward(self, x):
        return self.net(x)

class Receiver(nn.Module):
    """Scores each candidate object against the received message."""
    def __init__(self, msg_dim):
        super().__init__()
        self.msg_proj = nn.Linear(msg_dim, HIDDEN)
        self.obj_proj = nn.Linear(N_FEATURES, HIDDEN)
    def forward(self, msg, candidates):
        # msg:        (B, msg_dim)
        # candidates: (B, 10, N_FEATURES)
        mh = self.msg_proj(msg)                            # (B, HIDDEN)
        oh = self.obj_proj(candidates)                     # (B, 10, HIDDEN)
        scores = torch.bmm(oh, mh.unsqueeze(-1)).squeeze(-1)  # (B, 10)
        return scores

# ── Discrete training (Straight-Through Gumbel-Softmax, hard=True) ───────────
# This gives genuinely discrete tokens while remaining trainable.
# Temperature is annealed from 1.0 → 0.1 over training.

def train_discrete():
    sender   = Sender(output_dim=VOCAB_SIZE).to(device)
    receiver = Receiver(msg_dim=VOCAB_SIZE).to(device)
    opt      = torch.optim.Adam(list(sender.parameters()) +
                                list(receiver.parameters()), lr=LR)
    loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(N_EPOCHS):
        sender.train(); receiver.train()
        tau = max(0.1, 1.0 - 0.9 * epoch / N_EPOCHS)   # anneal temperature

        for si, lb, ri in loader:
            si, lb, ri = si.to(device), lb.to(device), ri.to(device)
            logits = sender(si)
            # straight-through: forward pass is hard one-hot, backward is soft
            msg = F.gumbel_softmax(logits, tau=tau, hard=True)
            scores = receiver(msg, ri)
            loss = F.cross_entropy(scores, lb)
            opt.zero_grad(); loss.backward(); opt.step()

        if (epoch+1) % 10 == 0:
            acc = eval_accuracy(sender, receiver, "discrete")
            print(f"[discrete] epoch {epoch+1:3d}  tau={tau:.2f}  val_acc={acc:.3f}")

    return sender, receiver

# ── Continuous training ────────────────────────────────────────────────────────
def train_continuous():
    sender   = Sender(output_dim=VOCAB_SIZE).to(device)
    # tanh to bound the message — fair comparison (discrete is naturally bounded)
    sender.net.add_module("tanh", nn.Tanh())
    receiver = Receiver(msg_dim=VOCAB_SIZE).to(device)
    opt      = torch.optim.Adam(list(sender.parameters()) +
                                list(receiver.parameters()), lr=LR)
    loader   = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    for epoch in range(N_EPOCHS):
        sender.train(); receiver.train()
        for si, lb, ri in loader:
            si, lb, ri = si.to(device), lb.to(device), ri.to(device)
            msg    = sender(si)
            scores = receiver(msg, ri)
            loss   = F.cross_entropy(scores, lb)
            opt.zero_grad(); loss.backward(); opt.step()

        if (epoch+1) % 10 == 0:
            acc = eval_accuracy(sender, receiver, "continuous")
            print(f"[continuous] epoch {epoch+1:3d}  val_acc={acc:.3f}")

    return sender, receiver

# ── Accuracy helper ───────────────────────────────────────────────────────────
def eval_accuracy(sender, receiver, protocol):
    sender.eval(); receiver.eval()
    accs = []
    with torch.no_grad():
        for si, lb, ri in DataLoader(val_ds, batch_size=256):
            si, lb, ri = si.to(device), lb.to(device), ri.to(device)
            raw = sender(si)
            msg = _make_message(raw, protocol, noise=0.0)
            scores = receiver(msg, ri)
            accs.append((scores.argmax(-1) == lb).float().mean().item())
    return float(np.mean(accs))

def _make_message(raw, protocol, noise):
    if protocol == "discrete":
        token = raw.argmax(dim=-1)
        if noise > 0:
            flip = torch.rand_like(token.float()) < noise
            rand_tokens = torch.randint(0, VOCAB_SIZE, token.shape, device=device)
            token = torch.where(flip, rand_tokens, token)
        return F.one_hot(token, num_classes=VOCAB_SIZE).float()
    else:
        return raw + torch.randn_like(raw) * noise

# ── Noise evaluation (THE KEY EXPERIMENT) ────────────────────────────────────
def evaluate_under_noise(sender, receiver, protocol):
    sender.eval(); receiver.eval()
    rows = []
    with torch.no_grad():
        for noise in NOISE_LEVELS:
            accs, entropies, wrong_ents, correct_ents = [], [], [], []
            for si, lb, ri in DataLoader(val_ds, batch_size=256):
                si, lb, ri = si.to(device), lb.to(device), ri.to(device)
                raw    = sender(si)
                msg    = _make_message(raw, protocol, noise)
                scores = receiver(msg, ri)
                probs  = F.softmax(scores, dim=-1)
                preds  = scores.argmax(-1)

                acc  = (preds == lb).float()
                ent  = -(probs * (probs.clamp(min=1e-9)).log()).sum(-1)

                accs.extend(acc.cpu().tolist())
                entropies.extend(ent.cpu().tolist())

                correct_mask = (preds == lb)
                wrong_mask   = ~correct_mask
                if correct_mask.any():
                    correct_ents.extend(ent[correct_mask].cpu().tolist())
                if wrong_mask.any():
                    wrong_ents.extend(ent[wrong_mask].cpu().tolist())

            row = {
                "noise":          noise,
                "accuracy":       round(float(np.mean(accs)), 4),
                "mean_entropy":   round(float(np.mean(entropies)), 4),
                "correct_entropy":round(float(np.mean(correct_ents)) if correct_ents else 0, 4),
                "wrong_entropy":  round(float(np.mean(wrong_ents))   if wrong_ents  else 0, 4),
            }
            rows.append(row)
            print(json.dumps({"protocol": protocol, **row}))
    return rows

# ── Run everything ────────────────────────────────────────────────────────────
print("="*50)
print("TRAINING DISCRETE")
print("="*50)
sender_d, receiver_d = train_discrete()

print("\n" + "="*50)
print("TRAINING CONTINUOUS")
print("="*50)
sender_c, receiver_c = train_continuous()

print("\n" + "="*50)
print("EVALUATING UNDER NOISE")
print("="*50)
results = {}
print("\n--- Discrete ---")
results["discrete"]   = evaluate_under_noise(sender_d, receiver_d, "discrete")
print("\n--- Continuous ---")
results["continuous"] = evaluate_under_noise(sender_c, receiver_c, "continuous")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for protocol, color in [("discrete", "steelblue"), ("continuous", "coral")]:
    rows = results[protocol]
    noise   = [r["noise"]         for r in rows]
    acc     = [r["accuracy"]      for r in rows]
    w_ent   = [r["wrong_entropy"] for r in rows]
    axes[0].plot(noise, acc,   marker='o', color=color, label=protocol)
    axes[1].plot(noise, w_ent, marker='o', color=color, label=protocol)

axes[0].set_title("Accuracy vs noise level")
axes[0].set_xlabel("Noise level"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title("Wrong-answer entropy vs noise level\n(higher = agent knows it's confused)")
axes[1].set_xlabel("Noise level"); axes[1].set_ylabel("Entropy on wrong predictions")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("experiment1_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nKey result to look for:")
print("Right plot: discrete line should RISE as noise increases (detects corruption)")
print("            continuous line should stay LOW even as it gets things wrong (silent failure)")

In [ ]:
"""
Experiment: Discreteness as Error Correction
=============================================
Core claim (from Deutsch + our analysis):
  Discrete communication channels allow agents to *detect* corrupted messages,
  whereas continuous channels degrade silently — the receiver cannot tell whether
  a difference in the received vector is meaningful signal or noise.

Setup:
  - Standard referential game: sender sees target object, sends message,
    receiver picks target from a lineup.
  - Two protocol variants trained identically:
      * DISCRETE  : sender outputs one token from a fixed vocab (REINFORCE)
      * CONTINUOUS: sender outputs a real-valued vector (straight-through GS / raw)
  - After training, we INJECT noise at varying levels (σ = 0, 0.1, … 1.0)
    and measure:
      (a) task accuracy  — does it still work?
      (b) error detectability — can the receiver tell something went wrong?
          Operationalised as: variance of receiver's output distribution.
          A receiver that *knows* it got garbage should be more uncertain (higher
          entropy / higher variance over candidates).  A receiver that silently
          degrades will stay confident but wrong.

This distinction is the testable version of the Deutsch argument.
"""

import argparse
import json
import os
from dataclasses import dataclass, field
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

# EGG core
import egg.core as core
from egg.core import Callback, ConsoleLogger, Interaction


# ---------------------------------------------------------------------------
# 1.  Data
# ---------------------------------------------------------------------------

def make_dataset(n_features: int, n_distractors: int, n_samples: int, seed: int = 42):
    """
    Each sample: a set of (n_distractors+1) objects represented as binary
    attribute vectors.  One is the target (index 0 after shuffle).
    Returns sender_input (target vector), labels (always 0), and full
    receiver_input (all candidates concatenated).
    """
    rng = np.random.default_rng(seed)
    objects = rng.integers(0, 2, size=(n_samples, n_distractors + 1, n_features)).astype(np.float32)
    sender_input = torch.tensor(objects[:, 0, :])          # target only
    labels       = torch.zeros(n_samples, dtype=torch.long) # target always at position 0
    # receiver sees all candidates (shuffled per sample so it can't cheat)
    receiver_input = torch.tensor(objects)                  # (N, n_dist+1, n_feat)
    return TensorDataset(sender_input, labels, receiver_input)


# ---------------------------------------------------------------------------
# 2.  Agents
# ---------------------------------------------------------------------------

class Sender(nn.Module):
    def __init__(self, n_features: int, hidden: int, vocab_size: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, vocab_size),
        )

    def forward(self, x, _aux_input=None):
        return self.fc(x)   # logits over vocab


class ContinuousSender(nn.Module):
    """Outputs a real-valued message vector instead of symbol logits."""
    def __init__(self, n_features: int, hidden: int, msg_dim: int):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, msg_dim),
            nn.Tanh(),          # bounded output
        )

    def forward(self, x, _aux_input=None):
        return self.fc(x)


class Receiver(nn.Module):
    def __init__(self, n_features: int, hidden: int, msg_dim: int, n_distractors: int):
        super().__init__()
        self.n_candidates = n_distractors + 1
        self.msg_embed = nn.Linear(msg_dim, hidden)
        self.obj_embed = nn.Linear(n_features, hidden)

    def forward(self, message, receiver_input, _aux_input=None):
        # receiver_input: (B, n_candidates, n_features)
        B = receiver_input.size(0)
        msg_h = self.msg_embed(msg)                          # (B, hidden)
        obj_h = self.obj_embed(receiver_input)               # (B, n_cand, hidden)
        # dot-product attention between message and each candidate
        scores = torch.bmm(obj_h, msg_h.unsqueeze(-1)).squeeze(-1)  # (B, n_cand)
        return scores   # raw logits; loss will apply cross-entropy


# ---------------------------------------------------------------------------
# 3.  Noise injection (applied AFTER training, during evaluation)
# ---------------------------------------------------------------------------

class NoisyChannelWrapper(nn.Module):
    """
    Wraps a trained game and injects additive Gaussian noise into the
    message BEFORE it reaches the receiver.  For discrete protocols the
    message is a one-hot / index, so we corrupt it differently:
    with probability `flip_prob` we replace the token with a random one.
    """
    def __init__(self, game, protocol: str, noise_level: float,
                 vocab_size: int = 0, msg_dim: int = 0):
        super().__init__()
        self.game = game
        self.protocol = protocol
        self.noise_level = noise_level
        self.vocab_size = vocab_size
        self.msg_dim = msg_dim

    def forward(self, *args, **kwargs):
        return self.game(*args, **kwargs)


def evaluate_with_noise(sender, receiver_module, dataset, protocol: str,
                        noise_level: float, vocab_size: int, msg_dim: int,
                        n_distractors: int, device: str = "cpu"):
    """
    Run evaluation injecting noise into the channel.
    Returns:
        accuracy         : float
        mean_entropy     : float  (receiver output distribution entropy)
        correct_entropy  : float  (entropy when prediction was correct)
        wrong_entropy    : float  (entropy when prediction was wrong)
    """
    sender.eval()
    receiver_module.eval()

    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    accs, entropies, correct_ents, wrong_ents = [], [], [], []

    with torch.no_grad():
        for sender_input, labels, receiver_input in loader:
            sender_input   = sender_input.to(device)
            labels         = labels.to(device)
            receiver_input = receiver_input.to(device)

            # --- sender produces message ---
            raw_msg = sender(sender_input)

            if protocol == "discrete":
                # argmax to get token, then one-hot embed for receiver
                token = raw_msg.argmax(dim=-1)           # (B,)

                # inject noise: flip token with probability noise_level
                flip_mask = torch.rand(token.size(), device=device) < noise_level
                random_tokens = torch.randint(0, vocab_size, token.size(), device=device)
                noisy_token = torch.where(flip_mask, random_tokens, token)

                # one-hot for receiver (msg_dim == vocab_size here)
                msg = F.one_hot(noisy_token, num_classes=vocab_size).float()

            else:  # continuous
                # inject additive Gaussian noise
                noise = torch.randn_like(raw_msg) * noise_level
                msg   = raw_msg + noise

            # --- receiver scores candidates ---
            scores = receiver_module(msg, None, receiver_input)   # (B, n_cand)
            probs  = F.softmax(scores, dim=-1)

            preds  = scores.argmax(dim=-1)
            acc    = (preds == labels).float()
            accs.extend(acc.cpu().tolist())

            # entropy of receiver distribution (higher = more uncertain)
            ent = -(probs * (probs + 1e-9).log()).sum(dim=-1)  # (B,)
            entropies.extend(ent.cpu().tolist())

            correct_mask = (preds == labels)
            if correct_mask.any():
                correct_ents.extend(ent[correct_mask].cpu().tolist())
            if (~correct_mask).any():
                wrong_ents.extend(ent[~correct_mask].cpu().tolist())

    return {
        "accuracy"       : float(np.mean(accs)),
        "mean_entropy"   : float(np.mean(entropies)),
        "correct_entropy": float(np.mean(correct_ents)) if correct_ents else 0.0,
        "wrong_entropy"  : float(np.mean(wrong_ents))   if wrong_ents  else 0.0,
    }


# ---------------------------------------------------------------------------
# 4.  Training helpers
# ---------------------------------------------------------------------------

def train_discrete(args, train_ds, val_ds, device):
    """Train with REINFORCE (hard discrete symbols)."""
    sender   = Sender(args.n_features, args.hidden, args.vocab_size).to(device)
    receiver = Receiver(args.n_features, args.hidden, args.vocab_size,
                        args.n_distractors).to(device)

    # EGG wrappers: SymbolGameReinforce handles the discrete bottleneck
    sender_w   = core.RnnSenderReinforce(
        sender,
        vocab_size=args.vocab_size,
        embed_dim=args.hidden,
        hidden_size=args.hidden,
        max_len=1,          # single-symbol message for clean comparison
        cell="rnn",
    )
    receiver_w = core.RnnReceiverDeterministic(
        receiver,
        vocab_size=args.vocab_size,
        embed_dim=args.hidden,
        hidden_size=args.hidden,
        cell="rnn",
    )

    def loss_fn(sender_input, message, receiver_input, receiver_output, labels, aux):
        # receiver_output is scores over candidates
        # but we need to pass receiver_input through; EGG passes it differently
        # so we just use receiver_output directly
        acc = (receiver_output.argmax(dim=-1) == labels).float().mean()
        loss = F.cross_entropy(receiver_output, labels)
        return loss, {"acc": acc}

    game = core.SenderReceiverRnnReinforce(
        sender_w, receiver_w, loss_fn,
        sender_entropy_coeff=0.01,
        receiver_entropy_coeff=0.001,
    )
    optimizer = torch.optim.Adam(game.parameters(), lr=args.lr)

    trainer = core.Trainer(
        game=game,
        optimizer=optimizer,
        train_data=DataLoader(train_ds, batch_size=args.batch_size, shuffle=True),
        validation_data=DataLoader(val_ds, batch_size=256),
        callbacks=[ConsoleLogger(as_json=True, print_train_loss=False)],
    )
    trainer.train(n_epochs=args.n_epochs)
    return sender_w, receiver_w, game


def train_continuous(args, train_ds, val_ds, device):
    """Train with continuous (real-valued) messages — no discrete bottleneck."""
    msg_dim  = args.vocab_size   # same dimensionality for fair comparison
    sender   = ContinuousSender(args.n_features, args.hidden, msg_dim).to(device)
    receiver = Receiver(args.n_features, args.hidden, msg_dim,
                        args.n_distractors).to(device)

    class ContinuousGame(nn.Module):
        def __init__(self, sender, receiver):
            super().__init__()
            self.sender   = sender
            self.receiver = receiver

        def forward(self, sender_input, labels, receiver_input, aux_input=None):
            msg    = self.sender(sender_input)
            scores = self.receiver(msg, None, receiver_input)
            loss   = F.cross_entropy(scores, labels)
            acc    = (scores.argmax(-1) == labels).float().mean()
            return loss, {"acc": acc}

    game      = ContinuousGame(sender, receiver)
    optimizer = torch.optim.Adam(game.parameters(), lr=args.lr)

    best_val  = 0.0
    for epoch in range(args.n_epochs):
        game.train()
        for sender_input, labels, receiver_input in DataLoader(
                train_ds, batch_size=args.batch_size, shuffle=True):
            optimizer.zero_grad()
            loss, info = game(sender_input, labels, receiver_input)
            loss.backward()
            optimizer.step()

        # quick val check
        game.eval()
        val_accs = []
        with torch.no_grad():
            for si, lb, ri in DataLoader(val_ds, batch_size=256):
                _, info = game(si, lb, ri)
                val_accs.append(info["acc"].item())
        val_acc = np.mean(val_accs)
        if (epoch + 1) % 10 == 0:
            print(json.dumps({"epoch": epoch+1, "protocol": "continuous",
                              "val_acc": round(val_acc, 4)}))

    return sender, receiver, game


# ---------------------------------------------------------------------------
# 5.  Main
# ---------------------------------------------------------------------------

def get_args():
    p = argparse.ArgumentParser()
    # EGG already registers: --batch_size, --lr, --vocab_size, --n_epochs,
    # --random_seed, --max_len.  We add only our own args here.
    p.add_argument("--n_features",    type=int,  default=16)
    p.add_argument("--n_distractors", type=int,  default=9)
    p.add_argument("--n_train",       type=int,  default=10000)
    p.add_argument("--n_val",         type=int,  default=2000)
    p.add_argument("--hidden",        type=int,  default=64)
    p.add_argument("--noise_levels",  type=str,  default="0,0.05,0.1,0.2,0.3,0.5,0.7,1.0")
    p.add_argument("--results_dir",   type=str,  default="results")
    p.add_argument("--protocol",      type=str,  default="both",
                   choices=["discrete", "continuous", "both"])
    return core.init(arg_parser=p)


def main():
    args = get_args()
    seed = args.random_seed if args.random_seed is not None else 42
    torch.manual_seed(seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    os.makedirs(args.results_dir, exist_ok=True)

    noise_levels = [float(x) for x in args.noise_levels.split(",")]

    # --- data ---
    train_ds = make_dataset(args.n_features, args.n_distractors, args.n_train, seed)
    val_ds   = make_dataset(args.n_features, args.n_distractors, args.n_val,   seed + 1)

    results = {}

    # --- DISCRETE ---
    if args.protocol in ("discrete", "both"):
        print("\n=== Training DISCRETE protocol ===")
        sender_d, receiver_d, _ = train_discrete(args, train_ds, val_ds, device)
        results["discrete"] = []
        for nl in noise_levels:
            metrics = evaluate_with_noise(
                sender_d, receiver_d, val_ds,
                protocol="discrete",
                noise_level=nl,
                vocab_size=args.vocab_size,
                msg_dim=args.vocab_size,
                n_distractors=args.n_distractors,
                device=device,
            )
            metrics["noise_level"] = nl
            results["discrete"].append(metrics)
            print(json.dumps({"protocol": "discrete", **metrics}))

    # --- CONTINUOUS ---
    if args.protocol in ("continuous", "both"):
        print("\n=== Training CONTINUOUS protocol ===")
        sender_c, receiver_c, _ = train_continuous(args, train_ds, val_ds, device)
        results["continuous"] = []
        for nl in noise_levels:
            metrics = evaluate_with_noise(
                sender_c, receiver_c, val_ds,
                protocol="continuous",
                noise_level=nl,
                vocab_size=args.vocab_size,
                msg_dim=args.vocab_size,
                n_distractors=args.n_distractors,
                device=device,
            )
            metrics["noise_level"] = nl
            results["continuous"].append(metrics)
            print(json.dumps({"protocol": "continuous", **metrics}))

    # --- save ---
    out_path = os.path.join(args.results_dir, "noise_results.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {out_path}")


if __name__ == "__main__":
    main()

In [ ]:
import sys
sys.argv = [
    "game.py",
    "--n_epochs", "50",
    "--protocol", "both",
    "--vocab_size", "32",
    "--lr", "1e-3",
    "--batch_size", "64",
    "--random_seed", "42"
]

# then call main()
main()